In [15]:
import requests


In [16]:
BASE_URL = "https://api.mangadex.org"
title = "chainsaw man"

# Search manga title

In [28]:
search_params = {
    "title": title,
    "limit": 10
}
r_search = requests.get(f"{BASE_URL}/manga", params=search_params)
    
manga = r_search.json().get("data", [])

In [45]:
manga_id = manga[0]['id']
manga_id

'a77742b1-befd-49a4-bff5-1ad4e6b0ef7b'

# Get chapters

In [46]:
feed_params = {
        "translatedLanguage[]": ["en"],
        "order[chapter]": "desc",
        "limit": 5
    }
r_feed = requests.get(f"{BASE_URL}/manga/{manga_id}/feed", params=feed_params)
chapters = r_feed.json().get("data", [])

In [47]:
chapter_id = chapters[0]['id'] #first chapter (latest chapter)'s id
chapters[0]

{'id': 'd9f9d6c4-7547-401e-9f1f-c604140ab716',
 'type': 'chapter',
 'attributes': {'volume': None,
  'chapter': '229',
  'title': 'Nightjar and Asa',
  'translatedLanguage': 'en',
  'externalUrl': 'https://mangaplus.shueisha.co.jp/viewer/1027694',
  'isUnavailable': False,
  'publishAt': '2026-02-17T15:05:39+00:00',
  'readableAt': '2026-02-17T15:05:39+00:00',
  'createdAt': '2026-02-17T15:05:38+00:00',
  'updatedAt': '2026-02-17T15:05:39+00:00',
  'version': 2,
  'pages': 0},
 'relationships': [{'id': '4f1de6a2-f0c5-4ac5-bce5-02c7dbb67deb',
   'type': 'scanlation_group'},
  {'id': 'a77742b1-befd-49a4-bff5-1ad4e6b0ef7b', 'type': 'manga'},
  {'id': '74d95af1-7492-4fca-bc44-10c9142703e8', 'type': 'user'}]}

# NOTE: If pages = 0 or theres an externalURL, it means the manga is not hosted on mangadex so we can't download the images!

In [48]:
def get_chapter_panels(id):
    # 1. Ask MangaDex which server to use
    r = requests.get(f"https://api.mangadex.org/at-home/server/{id}")
    data = r.json()
    
    # 2. Grab the base URL and the chapter-specific hash
    base_url = data["baseUrl"]
    chapter_hash = data["chapter"]["hash"]
    
    # 'data' contains the high-quality filenames
    # 'dataSaver' contains the compressed/smaller filenames
    file_names = data["chapter"]["data"] 
    
    # 3. Construct the full URL for every page
    # Format: {baseUrl}/data/{hash}/{filename}
    urls = [f"{base_url}/data/{chapter_hash}/{name}" for name in file_names]
    
    return urls

In [49]:
panel_urls = get_chapter_panels(chapter_id)
panel_urls[:3]

[]

In [42]:
first_panel = panel_urls[0]
first_panel

'https://cmdxd98sb0x3yprd.mangadex.network/data/0153f7c6f0c42c2ae95fbee6284a1814/1-895cf8473ce580800d5d3f6b50bac68f8f2eae7b2395f610b51f5a2c8fe5a913.jpg'

# downloading the chapter locally

In [43]:
def download_panel(url, filename="manga_panel.jpg"):
    # 1. Send a GET request to the image URL
    response = requests.get(url, stream=True)
    
    if response.status_code == 200:
        # 2. Open a local file in 'wb' (write binary) mode
        with open(filename, 'wb') as f:
            for chunk in response.iter_content(1024):
                f.write(chunk)
        print(f"Success! Saved as {filename}")
    else:
        print(f"Failed to download. Status code: {response.status_code}")

In [44]:
download_panel(first_panel, "chainsaw_man_p1.jpg")

Success! Saved as chainsaw_man_p1.jpg


# end mangadex api testing

In [50]:
import mangaplus

In [ ]:
mangaplus.MangaPlus

In [ ]:
chapter_url = 'https://mangaplus.shueisha.co.jp/viewer/1027694'

def download_mangaplus_panel(url):
    # 1. Initialize the MangaPlus API client
    api = mangaplus.MangaPlus()
    
    # 2. Extract the chapter ID from the URL (1027694)
    chapter_id = url.split('/')[-1]
    
    # 3. Get the chapter details (this handles the Protobuf decryption)
    chapter = api.getMangaData(chapter_id=chapter_id)
    
    # 4. Get the URL for the first page
    # MangaPlus images are protected, so we use the library's downloader
    first_page = chapter.pages[0]
    
    # This specifically downloads AND decrypts the image
    image_data = api.download_image(first_page.manga_page.image_url, first_page.manga_page.encryption_key)
    
    # 5. Save the decrypted bytes to a file
    with open("mangaplus_panel_decrypted.jpg", "wb") as f:
        f.write(image_data)
    
    print("Success! The decrypted panel is saved as mangaplus_panel_decrypted.jpg")

download_mangaplus_panel(chapter_url)

AttributeError: 'MangaPlus' object has no attribute 'get_chapter_details'